# 08 Phase A And B Benchmark Comparison

This notebook consolidates the corrected-horizon benchmark stack:
- official naive benchmark
- ARIMA
- SARIMA
- LEAR `FS1` and `FS2`
- XGBoost `FS1` and `FS2`

This is the notebook that re-establishes `FS2` as the clean feature-based benchmark reference under the corrected `D..D+4` horizon.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig, MonitoringConfig
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.notebook_support import (
    build_week_metrics_for_predictions,
    build_reporting_metric_grid,
    estimate_run_duration_seconds,
    format_duration,
    load_selected_case_weeks,
    render_plot_gallery,
    render_reporting_metric_dashboard,
    run_suite_with_feedback,
    style_reporting_metric_grid,
    summarize_timing_compact,
    write_week_plots_for_models,
)

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
run_root = output_root / "runs"


def latest_run_matching(pattern: str) -> Path:
    matches = sorted(run_root.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No run directories found for pattern: {pattern}")
    return matches[-1]


This notebook is load-first by design. The corrected-horizon benchmark comparison should normally be read from the saved artifacts, because it is the checkpoint the later phased work depends on.


In [ ]:
run_dir = find_latest_run(output_root, "model_comparison")
print(run_dir)

metrics_overall = load_csv(run_dir, "metrics_overall.csv")
metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
predictions_long = load_csv(run_dir, "predictions_long.csv")
diebold_mariano_by_reporting_level = load_csv(run_dir, "diebold_mariano_by_reporting_level.csv")
official_naive = load_json(run_dir, "official_naive_reference.json")
source_runs = load_json(run_dir, "source_runs.json")


In [ ]:
display(pd.DataFrame([official_naive]))
display(pd.DataFrame(source_runs["source_runs"]))

display(
    metrics_by_reporting_level.sort_values(["dataset_split", "reporting_level_sort_order", "mae", "model"]).reset_index(drop=True)
)

display(
    diebold_mariano_by_reporting_level.sort_values(["dataset_split", "reporting_level_sort_order", "challenger_model"]).reset_index(drop=True)
)


In [ ]:
selection_run_dir, selected_weeks = load_selected_case_weeks(output_root)
print(selection_run_dir)
display(selected_weeks[["category", "iso_week_id", "week_start_local_date", "week_end_local_date"]])

week_metrics = build_week_metrics_for_predictions(predictions_long, config, selected_weeks)
display(
    week_metrics[week_metrics["model"].isin(['naive_previous_week', 'sarima', 'lear_fs2', 'xgboost_fs2'])]
    .sort_values(["category", "mae", "model"])
    .reset_index(drop=True)
)

plot_dir = output_root / "notebook_artifacts" / "08_phase_ab_benchmark_comparison" / run_dir.name if "run_dir" in globals() else output_root / "notebook_artifacts" / "08_phase_ab_benchmark_comparison" / phase_run_dir.name
plot_paths = write_week_plots_for_models(
    predictions=predictions_long,
    config=config,
    selected_weeks=selected_weeks,
    output_dir=plot_dir,
    models=['naive_previous_week', 'sarima', 'lear_fs2', 'xgboost_fs2'],
    title_prefix="Phase A/B benchmark comparison",
)
display(render_plot_gallery(plot_paths, columns=2))
